In [1]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
 


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [2]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

In [3]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [4]:
import pandas as pd
import numpy as np
bucket_name ='ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

df = pd.read_csv('s3://{}/{}/REPLICA_OUTPUT/scr_PLAFPJMINORISTA_202609.txt'.format(bucket_name, model_prefix ), sep= '|')

In [5]:
import pandas as pd
import numpy as np
bucket_name ='ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

df_1 = pd.read_csv('s3://{}/{}/REPLICA_OUTPUT/scr_PLAFPJMINORISTA_202508.txt'.format(bucket_name, model_prefix ), sep= '|')

In [6]:
df.head(10)

,cod_mes,cod_tip_doc,key_value,cod_cli,prob_model,puntuacion,modelo,ts_carga,grupo_ejec,segmento,orden,extra_01,extra_02,extra_03,extra_04,extra_05
0,202609,2.0,5183F9EE96E26A089F5605A2AB709165B31E1D3F369575...,20486403,NaN,0.992008,plaft_pj_minorista,20260911,P1,NaN,1,NaN,SIN_INFO,0,NaN,NaN
1,202609,2.0,96ACB8B73724FD10F705B7F6F9B0F4755282E6780E575B...,21248487,NaN,0.989526,plaft_pj_minorista,20260911,P1,NaN,2,NaN,SIN_INFO,0,NaN,NaN
2,202609,2.0,451E1CB29AACDEF08EBE73D26D6E201DD278319B61B354...,17172897,NaN,0.985868,plaft_pj_minorista,20260911,P1,NaN,3,NaN,SIN_INFO,0,NaN,NaN
3,202609,2.0,AF48682B5D2FECE0CD8CD5ABC211E7C5A935B26F5FEF10...,20333816,NaN,0.984694,plaft_pj_minorista,20260911,P1,NaN,4,NaN,SIN_INFO,0,NaN,NaN
4,202609,2.0,3B71100C12E77007809973AC991B81298E8F1F4FDA2A57...,18352839,NaN,0.983374,plaft_pj_minorista,20260911,P1,NaN,5,NaN,SIN_INFO,0,NaN,NaN
5,202609,2.0,5C92ECF168AE1EF088EE8F0EA04AE8C86E037B06042760...,21711626,NaN,0.983124,plaft_pj_minorista,20260911,P1,NaN,6,NaN,SIN_INFO,0,NaN,NaN
6,202609,2.0,44AE54BD5414047AA109B2515D39D788B1719D48B4E76A...,22006259,NaN,0.982855,plaft_pj_minorista,20260911,P1,NaN,7,NaN,SIN_INFO,0,NaN,NaN
7,202609,2.0,98F04B049946F3FDF70D81AE315C6ACBDE816A252A77F7...,14566239,NaN,0.982495,plaft_pj_minorista,20260911,P1,NaN,8,NaN,SIN_INFO,0,NaN,NaN
8,202609,2.0,A63682D09F8E57F75790D120F5E497AC6093A1CE7F9EBF...,21866727,NaN,0.981708,plaft_pj_minorista,20260911,P1,NaN,9,NaN,SIN_INFO,0,NaN,NaN
9,202609,2.0,4E14146806A5EB55AE3C1336C3280E305A1BA13305B8EF...,13384198,NaN,0.981673,plaft_pj_minorista,20260911,P1,NaN,10,NaN,SIN_INFO,0,NaN,NaN


In [7]:
df.variable1.value_counts()

AttributeError: 'DataFrame' object has no attribute 'variable1'

In [8]:
df.head(10)

,cod_mes,cod_tip_doc,key_value,cod_cli,prob_model,puntuacion,modelo,ts_carga,grupo_ejec,segmento,orden,extra_01,extra_02,extra_03,extra_04,extra_05
0,202609,2.0,5183F9EE96E26A089F5605A2AB709165B31E1D3F369575...,20486403,NaN,0.992008,plaft_pj_minorista,20260911,P1,NaN,1,NaN,SIN_INFO,0,NaN,NaN
1,202609,2.0,96ACB8B73724FD10F705B7F6F9B0F4755282E6780E575B...,21248487,NaN,0.989526,plaft_pj_minorista,20260911,P1,NaN,2,NaN,SIN_INFO,0,NaN,NaN
2,202609,2.0,451E1CB29AACDEF08EBE73D26D6E201DD278319B61B354...,17172897,NaN,0.985868,plaft_pj_minorista,20260911,P1,NaN,3,NaN,SIN_INFO,0,NaN,NaN
3,202609,2.0,AF48682B5D2FECE0CD8CD5ABC211E7C5A935B26F5FEF10...,20333816,NaN,0.984694,plaft_pj_minorista,20260911,P1,NaN,4,NaN,SIN_INFO,0,NaN,NaN
4,202609,2.0,3B71100C12E77007809973AC991B81298E8F1F4FDA2A57...,18352839,NaN,0.983374,plaft_pj_minorista,20260911,P1,NaN,5,NaN,SIN_INFO,0,NaN,NaN
5,202609,2.0,5C92ECF168AE1EF088EE8F0EA04AE8C86E037B06042760...,21711626,NaN,0.983124,plaft_pj_minorista,20260911,P1,NaN,6,NaN,SIN_INFO,0,NaN,NaN
6,202609,2.0,44AE54BD5414047AA109B2515D39D788B1719D48B4E76A...,22006259,NaN,0.982855,plaft_pj_minorista,20260911,P1,NaN,7,NaN,SIN_INFO,0,NaN,NaN
7,202609,2.0,98F04B049946F3FDF70D81AE315C6ACBDE816A252A77F7...,14566239,NaN,0.982495,plaft_pj_minorista,20260911,P1,NaN,8,NaN,SIN_INFO,0,NaN,NaN
8,202609,2.0,A63682D09F8E57F75790D120F5E497AC6093A1CE7F9EBF...,21866727,NaN,0.981708,plaft_pj_minorista,20260911,P1,NaN,9,NaN,SIN_INFO,0,NaN,NaN
9,202609,2.0,4E14146806A5EB55AE3C1336C3280E305A1BA13305B8EF...,13384198,NaN,0.981673,plaft_pj_minorista,20260911,P1,NaN,10,NaN,SIN_INFO,0,NaN,NaN


In [14]:
from pathlib import Path

ruta_salida_df = Path.cwd() / 'df_renta_alta_202607.csv'
df.to_csv(ruta_salida_df, index=False, encoding='utf-8-sig')

print(f'DataFrame guardado en: {ruta_salida_df}')
print(f'Filas: {len(df):,} | Columnas: {len(df.columns):,}')

DataFrame guardado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Minorista_regulado\Inferencia\df_renta_alta_202607.csv
Filas: 180,309 | Columnas: 12


In [9]:
df_res = (
    df
    .groupby(['grupo_alerta', 'codmes'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_alerta', columns='codmes', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

print(df_res)

KeyError: 'grupo_alerta'

In [16]:
import pandas as pd

# === Tu pivot original (cantidades) ===
df_res = (
    df
    .groupby(['grupo_alerta', 'codmes'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_alerta', columns='codmes', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

# === PORCENTAJES POR MES (suman 100% por columna / por mes) ===
df_months = df_res.iloc[:, :-1]                     # todas las columnas de meses (sin TOTAL)
month_totals = df_months.sum(axis=0)                # total por cada mes

df_pct = (df_months.div(month_totals, axis=1) * 100).round(1)
df_pct = df_pct.astype(str) + '%'

# Renombrar columnas para que quede claro
df_pct.columns = [f'{col} %' for col in df_pct.columns]

# === Mostrar resultados ===
print("=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===")
print(df_res)

print("\n=== PORCENTAJES por mes (cada columna suma 100%) ===")
print(df_pct)



=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===
codmes        202609   TOTAL
grupo_alerta                
P5            175653  175653
P4              3982    3982
P3               587     587
P2                86      86
P1                 1       1

=== PORCENTAJES por mes (cada columna suma 100%) ===
             202609 %
grupo_alerta         
P5              97.4%
P4               2.2%
P3               0.3%
P2               0.0%
P1               0.0%


In [62]:
df_final_alerta= df[(df.variable2 !='SIN_INFO')]

In [63]:
#test_data_0825_alerta_1= test_data_0825_alerta[(test_data_0825_alerta.variable1 !=0)]

In [64]:
import pandas as pd

# === Tu pivot original (cantidades) ===
df_res = (
    df_final_alerta
    .groupby(['grupo_alerta', 'codmes'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_alerta', columns='codmes', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

# === PORCENTAJES POR MES (suman 100% por columna / por mes) ===
df_months = df_res.iloc[:, :-1]                     # todas las columnas de meses (sin TOTAL)
month_totals = df_months.sum(axis=0)                # total por cada mes

df_pct = (df_months.div(month_totals, axis=1) * 100).round(1)
df_pct = df_pct.astype(str) + '%'

# Renombrar columnas para que quede claro
df_pct.columns = [f'{col} %' for col in df_pct.columns]

# === Mostrar resultados ===
print("=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===")
print(df_res)

print("\n=== PORCENTAJES por mes (cada columna suma 100%) ===")
print(df_pct)

=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===
codmes        202508  TOTAL
grupo_alerta               
P2               160    160
P1               152    152
P3               121    121
P4                68     68
P5                53     53

=== PORCENTAJES por mes (cada columna suma 100%) ===
             202508 %
grupo_alerta         
P2              28.9%
P1              27.4%
P3              21.8%
P4              12.3%
P5               9.6%


In [65]:
%%time
query = """

WITH pd AS (
    SELECT DISTINCT
        -- Identificadores
        a.num_documento,
        a.codunico,
        date_format(
            date_parse(CAST(a.mes_base AS varchar), '%Y%m') + interval '1' month,
            '%Y%m'
        ) AS codmes_lag1,
        CAST(a.mes_base AS INTEGER) AS codmes
    

    FROM d_perm_aws.ds_alertplaft_mdl a
    WHERE a.mes_base =   '202508' 
      AND a.subsegmento = 'BPE'
),

target AS (
    SELECT 
        codunico,
        periodo_alerta,
        tipo_alerta_n2,
        MAX(calificacion_monitoreo) AS flg_alerta
    FROM e_perm_aws.t_alertas_plaft
    GROUP BY codunico, periodo_alerta, tipo_alerta_n2
)

SELECT 
    a.*,
    b.tipo_alerta_n2,
    CASE 
        WHEN b.flg_alerta = '1' THEN 1 
        ELSE 0 
    END AS target_m
FROM pd a
LEFT JOIN target b
    ON a.codunico = b.codunico
 AND cast(a.codmes as varchar) = b.periodo_alerta
-- AND codmes_lag1 = b.periodo_alerta
;"""

df_dataset = athena_query(query, database='disc_comercial')
df_dataset.head()

CPU times: total: 1.73 s
Wall time: 15.5 s


,num_documento,codunico,codmes_lag1,codmes,tipo_alerta_n2,target_m
0,3CB00115E2E4C43EA65ACFD20AF31F2A7A6817E1DBA001...,0021489304,202509,202508,<NA>,0
1,0840565BB1A943694D4A5D2E03A5C4C994843B2E089CF6...,0018220889,202509,202508,<NA>,0
2,DF23D9F78ED88B5CA1B562106ABBAAC53918D842914481...,0021278311,202509,202508,<NA>,0
3,D8CBD9462CA6B8A4530139A612EC61EB89E287DA32ABDD...,0013678234,202509,202508,<NA>,0
4,25C9F45B076B092F497A435BA29B6C2DE6683E36A9E30E...,0020625928,202509,202508,<NA>,0


In [66]:
df_dataset.target_m.value_counts()

target_m
0    161313
1       122
Name: count, dtype: Int64

In [67]:
import pandas as pd

# Aseguramos tipos consistentes
df['codmes'] = df['codmes'].astype(str)
df_dataset['codmes'] = df_dataset['codmes'].astype(str)

df['codunico'] = pd.to_numeric(df['codunico'], errors='coerce').astype('Int64')
df_dataset['codunico'] = pd.to_numeric(df_dataset['codunico'], errors='coerce').astype('Int64')

# 1. Cross join parcial (o merge sin condición de mes primero)
df_4_temp = pd.merge(
    df,
    df_dataset,
    on=['num_documento', 'codunico'],
    how='inner',
    suffixes=('_final', '_dataset')
)

# 2. Filtramos solo las filas donde codmes_dataset >= codmes_final
df_1 = df_4_temp[
    df_4_temp['codmes_dataset'] >= df_4_temp['codmes_final']
].copy()

# 3. (Opcional) Si hay múltiples matches por cliente-mes, puedes quedarte con el primero o el más cercano
# Ejemplo: ordenar por diferencia de mes y quedarte con el menor (más cercano al actual)
df_1['mes_diff'] = pd.to_numeric(df_1['codmes_dataset']) - pd.to_numeric(df_1['codmes_final'])

# Ordenar por cliente + mes_diff ascendente (el más cercano primero)
df_1 = df_1.sort_values(['num_documento', 'codunico', 'mes_diff'])

# Opcional: quedarte solo con el match más cercano por cada fila de df_final
df_1 = df_1.drop_duplicates(subset=['num_documento', 'codunico', 'codmes_final'], keep='first')

# Limpiar columnas innecesarias si quieres
df_1 = df_1.drop(columns=['mes_diff'], errors='ignore')

In [68]:
df_1.head()

,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m
70230,202508,0000081DD593FDADA1F996DE30B1B0323EC1EF690041DA...,19985965,plaft_pj_minorista,20260901,P4,0,0.631293,70213,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
106617,202508,00000E38993C51A1DD6E4048C653C80A2AFB66E34527AB...,20354392,plaft_pj_minorista,20260901,P4,0,0.469248,106600,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
141640,202508,0000BCAB0002120120BC97DA0CF7D372987995159FD0A3...,910935,plaft_pj_minorista,20260901,P4,0,0.235446,141622,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
85596,202508,0000E5A1B73363AEFCAA055EB76D9AE4A7F1C7F46E82D2...,19813872,plaft_pj_minorista,20260901,P4,0,0.565066,85579,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
79641,202508,00015A2F62D6AC753C86D623C0A7BF75BBD6FDF89A909C...,13337711,plaft_pj_minorista,20260901,P4,0,0.593227,79624,NaN,SIN_INFO,NaN,202509,202508,<NA>,0


In [70]:
df_1.target_m.value_counts()

target_m
0    161296
1       119
Name: count, dtype: Int64

In [71]:
df_1.variable1.value_counts()

variable1
AUTOMATICA         424
MANUAL              72
SEMI AUTOMATICA     58
Name: count, dtype: int64

In [72]:
df_1.grupo_corte_nueva_alerta.value_counts()

grupo_corte_nueva_alerta
0    161415
Name: count, dtype: int64

In [73]:
df_res = (
    df_1
    .groupby(['grupo_alerta', 'codmes_final'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_alerta', columns='codmes_final', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

print(df_res)

codmes_final  202508  TOTAL
grupo_alerta               
P4             86721  86721
P3             48217  48217
P5             18974  18974
P2              6395   6395
P1              1108   1108


In [74]:
df_2= df_1[(df_1.grupo_corte_nueva_alerta ==0)]

In [75]:
df_res = (
    df_2
    .groupby(['grupo_alerta', 'codmes_final'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_alerta', columns='codmes_final', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

print(df_res)

codmes_final  202508  TOTAL
grupo_alerta               
P4             86721  86721
P3             48217  48217
P5             18974  18974
P2              6395   6395
P1              1108   1108


In [76]:
# Solo alertas  

In [77]:
df_1.head()

,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m
70230,202508,0000081DD593FDADA1F996DE30B1B0323EC1EF690041DA...,19985965,plaft_pj_minorista,20260901,P4,0,0.631293,70213,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
106617,202508,00000E38993C51A1DD6E4048C653C80A2AFB66E34527AB...,20354392,plaft_pj_minorista,20260901,P4,0,0.469248,106600,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
141640,202508,0000BCAB0002120120BC97DA0CF7D372987995159FD0A3...,910935,plaft_pj_minorista,20260901,P4,0,0.235446,141622,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
85596,202508,0000E5A1B73363AEFCAA055EB76D9AE4A7F1C7F46E82D2...,19813872,plaft_pj_minorista,20260901,P4,0,0.565066,85579,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
79641,202508,00015A2F62D6AC753C86D623C0A7BF75BBD6FDF89A909C...,13337711,plaft_pj_minorista,20260901,P4,0,0.593227,79624,NaN,SIN_INFO,NaN,202509,202508,<NA>,0


In [78]:
df_3= df_1[(df_1.variable2 !='SIN_INFO')]

In [79]:
df_3.head()

,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m
146697,202508,000FE24299055E443CA832AE6DB6C0B6302006903802C9...,16347064,plaft_pj_minorista,20260901,P5,0,0.161973,146679,AUTOMATICA,M,NaN,202509,202508,AUTOMATICA,0
135392,202508,003314A509A3C4A206D7EE3EC23D4EF2836D826509A416...,19992688,plaft_pj_minorista,20260901,P4,0,0.302118,135374,AUTOMATICA,M,NaN,202509,202508,AUTOMATICA,0
2695,202508,0039E83BD5BB9284BB4AB6F7884615DFE0C3C206863614...,17568521,plaft_pj_minorista,20260901,P2,0,0.950082,2680,AUTOMATICA,A,NaN,202509,202508,AUTOMATICA,0
129,202508,00E5CF5595BCE6D77598FEE3DA207F5352FB52D7D58EAA...,21563570,plaft_pj_minorista,20260901,P1,0,0.993685,124,AUTOMATICA,A,NaN,202509,202508,AUTOMATICA,1
19647,202508,010634C31E9CCF394B9840C31A94407612D5E1B632DECF...,19373694,plaft_pj_minorista,20260901,P3,0,0.822219,19630,AUTOMATICA,A,NaN,202509,202508,AUTOMATICA,0


In [80]:
# Resumen por grupo_alerta: total de casos y casos positivos (target_m = 1)
if 'target_m' not in df_3.columns:
    raise ValueError("df_3 no tiene la columna target_m")

resumen_grupo = (
    df_3.assign(target_m_num=pd.to_numeric(df_3['target_m'], errors='coerce').fillna(0).astype(int))
       .groupby('grupo_alerta', as_index=False)
       .agg(
           total_casos=('target_m_num', 'size'),
           casos_positivos=('target_m_num', 'sum')
       )
       .sort_values('grupo_alerta')
)

resumen_grupo['tasa_positivos'] = (resumen_grupo['casos_positivos'] / resumen_grupo['total_casos']).round(4)

print("=== Casos por grupo_alerta ===")
display(resumen_grupo)

=== Casos por grupo_alerta ===


,grupo_alerta,total_casos,casos_positivos,tasa_positivos
0,P1,152,68,0.4474
1,P2,160,36,0.2250
2,P3,121,13,0.1074
3,P4,68,2,0.0294
4,P5,53,0,0.0000


In [51]:
# Resumen por tipo_alerta_n2: total de casos y casos positivos (target_m = 1)
if 'tipo_alerta_n2' not in df_3.columns:
    raise ValueError("df_3 no tiene la columna tipo_alerta_n2")
if 'target_m' not in df_3.columns:
    raise ValueError("df_3 no tiene la columna target_m")

resumen_tipo_alerta_n2 = (
    df_3.assign(
        tipo_alerta_n2=df_3['tipo_alerta_n2'].fillna('SIN_TIPO_ALERTA'),
        target_m_num=pd.to_numeric(df_3['target_m'], errors='coerce').fillna(0).astype(int)
    )
    .groupby('tipo_alerta_n2', as_index=False)
    .agg(
        total_casos=('target_m_num', 'size'),
        casos_positivos=('target_m_num', 'sum')
    )
    .sort_values('total_casos', ascending=False)
    .reset_index(drop=True)
)

resumen_tipo_alerta_n2['tasa_positivos'] = (
    resumen_tipo_alerta_n2['casos_positivos'] / resumen_tipo_alerta_n2['total_casos']
).round(4)

print("=== Casos por tipo_alerta_n2 ===")
display(resumen_tipo_alerta_n2)

=== Casos por tipo_alerta_n2 ===


,tipo_alerta_n2,total_casos,casos_positivos,tasa_positivos
0,AUTOMATICA,424,59,0.1392
1,MANUAL,73,37,0.5068
2,SEMI AUTOMATICA,57,23,0.4035


In [52]:
from pathlib import Path
import pandas as pd

# ====================================================================
# Exportar HTML mensual con APERTURA POR variable1 (mismo formato tríptico)
# ====================================================================
if 'df_3' not in globals():
    raise ValueError("No existe df_3 en memoria.")

required_cols = {'variable1', 'grupo_alerta', 'target_m'}
missing = required_cols.difference(df_3.columns)
if missing:
    raise ValueError(f"Faltan columnas requeridas en df_3: {missing}")

# Detectar periodo
period_cols = ['codmes_final', 'codmes', 'mes_base']
period_value = None
for col in period_cols:
    if col in df_3.columns:
        s = pd.to_numeric(df_3[col], errors='coerce').dropna().astype(int).astype(str)
        s = s[s.str.len() >= 6]
        if not s.empty:
            period_value = s.max()[:6]
            break
if period_value is None:
    period_value = '202602'

anio = period_value[:4]
mes_num = period_value[4:6]
meses_es = {
    '01': 'enero', '02': 'febrero', '03': 'marzo', '04': 'abril',
    '05': 'mayo', '06': 'junio', '07': 'julio', '08': 'agosto',
    '09': 'septiembre', '10': 'octubre', '11': 'noviembre', '12': 'diciembre'
}
mes_nombre = meses_es.get(mes_num, f"mes_{mes_num}")
mes_label = f"{mes_nombre.capitalize()} {anio}"

work_df = df_3.copy()
work_df['tipo_det'] = (
    work_df['variable1'].astype(str).str.replace('_', ' ', regex=False).str.strip().str.upper()
)
work_df['target_m_num'] = pd.to_numeric(work_df['target_m'], errors='coerce').fillna(0).astype(int)

panel_cfg = [
    {'key': 'AUTOMATICA', 'title': 'AUTOMÁTICA', 'icon': 'fas fa-robot fa-lg', 'header': 'header-auto', 'bar_color': 'var(--interbank-azul)'},
    {'key': 'MANUAL', 'title': 'MANUAL', 'icon': 'fas fa-hand-holding-usd fa-lg', 'header': 'header-manual', 'bar_color': 'var(--interbank-verde-claro)'},
    {'key': 'SEMI AUTOMATICA', 'title': 'SEMI AUTOMÁTICA', 'icon': 'fas fa-cogs fa-lg', 'header': 'header-semi', 'bar_color': 'var(--interbank-verde-oscuro)'},
]

sum_tipo = (
    work_df.groupby('tipo_det', as_index=False)
    .agg(alertas=('target_m_num', 'size'), riesgo=('target_m_num', 'sum'))
)
max_alertas = max(int(sum_tipo['alertas'].max()), 1) if not sum_tipo.empty else 1

def volumen_label(width_pct: float) -> str:
    if width_pct >= 66: return 'Alto'
    if width_pct >= 33: return 'Medio'
    return 'Bajo'

def fmt_pct(x: float) -> str:
    return f"{x*100:.2f}%"

def build_rows(df_tipo: pd.DataFrame) -> str:
    grp = (
        df_tipo.groupby('grupo_alerta', as_index=False)
        .agg(alertas=('target_m_num', 'size'), riesgo=('target_m_num', 'sum'))
    )
    if grp.empty:
        return "<tr><td colspan='4' class='text-center'>Sin datos</td></tr>"
    grp['efect'] = (grp['riesgo'] / grp['alertas']).fillna(0)
    grp = grp.sort_values('grupo_alerta')

    rows = []
    for _, r in grp.iterrows():
        riesgo_cls = "text-center text-red-600 font-bold" if int(r['riesgo']) > 0 else "text-center"
        efect_cls = "text-right font-bold text-green-600" if r['efect'] >= 0.5 else "text-right font-bold"
        rows.append(
            f"<tr>"
            f"<td><span class='group-badge'>{r['grupo_alerta']}</span></td>"
            f"<td class='text-center'>{int(r['alertas'])}</td>"
            f"<td class='{riesgo_cls}'>{int(r['riesgo'])}</td>"
            f"<td class='{efect_cls}'>{fmt_pct(r['efect'])}</td>"
            f"</tr>"
        )
    return ''.join(rows)

panels_html = []
stats_by_panel = []
for cfg in panel_cfg:
    df_tipo = work_df[work_df['tipo_det'] == cfg['key']].copy()
    alertas = int(len(df_tipo))
    riesgo = int(df_tipo['target_m_num'].sum()) if not df_tipo.empty else 0
    efect = (riesgo / alertas) if alertas > 0 else 0.0
    width_pct = (alertas / max_alertas) * 100 if max_alertas > 0 else 0
    vol_lbl = volumen_label(width_pct)
    rows_html = build_rows(df_tipo)

    panel_html = f"""
    <div class=\"panel\">
      <div class=\"panel-header {cfg['header']}\">
        <i class=\"{cfg['icon']}\"></i>
        <h2>{cfg['title']}</h2>
      </div>
      <div class=\"kpi-grid\">
        <div class=\"kpi-card\"><p class=\"kpi-label\">Alertas</p><p class=\"kpi-value\">{alertas}</p></div>
        <div class=\"kpi-card\"><p class=\"kpi-label\">Riesgo</p><p class=\"kpi-value risk\">{riesgo}</p></div>
        <div class=\"kpi-card full-width\"><p class=\"kpi-label\">Efectividad Global</p><p class=\"kpi-value success\">{fmt_pct(efect)}</p></div>
      </div>
      <div class=\"table-section\">
        <h3 class=\"section-title\">Desagregación por Grupo</h3>
        <table class=\"data-table\">
          <thead><tr><th>Grupo</th><th class=\"text-center\">Alertas</th><th class=\"text-center\">Riesgo</th><th class=\"text-right\">Efect.</th></tr></thead>
          <tbody>{rows_html}</tbody>
        </table>
        <div style=\"margin-top: auto; padding-top: 10px;\">
          <div class=\"volume-label\" style=\"display: flex; justify-content: space-between; margin-bottom: 2px;\"><span>Volumen</span><span>{vol_lbl}</span></div>
          <div class=\"bar-container\"><div class=\"bar-fill\" style=\"width: {width_pct:.1f}%; background-color: {cfg['bar_color']};\"></div></div>
        </div>
      </div>
    </div>
    """
    panels_html.append(panel_html)
    stats_by_panel.append({'title': cfg['title'], 'alertas': alertas, 'efect': efect})

if stats_by_panel:
    top_efect = max(stats_by_panel, key=lambda x: x['efect'])
    top_vol = max(stats_by_panel, key=lambda x: x['alertas'])
    insight_txt = (
        f"La detección {top_efect['title']} muestra la mayor efectividad ({fmt_pct(top_efect['efect'])}), "
        f"mientras que {top_vol['title']} concentra el mayor volumen de alertas ({top_vol['alertas']})."
    )
else:
    insight_txt = "No hay datos disponibles para construir el insight."

html = f"""
<!DOCTYPE html>
<html lang=\"es\">
<head>
  <meta charset=\"utf-8\"/>
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\"/>
  <title>Análisis por Tipo de Detección - Interbank</title>
  <link href=\"https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap\" rel=\"stylesheet\"/>
  <link href=\"https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css\" rel=\"stylesheet\"/>
  <link href=\"https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.4.0/css/all.min.css\" rel=\"stylesheet\"/>
  <style>
    :root {{ --interbank-verde-oscuro: rgb(0,95,30); --interbank-verde-claro: rgb(0,190,80); --interbank-verde-suave: rgb(198,255,192); --interbank-azul: rgb(31,69,146); --bg-gray:#f8f9fa; }}
    body {{ font-family:'Poppins',sans-serif; background-color:white; margin:0; padding:0; width:1280px; height:720px; overflow:hidden; }}
    .container {{ width:1280px; height:720px; position:relative; padding:20px 40px; display:flex; flex-direction:column; }}
    .header-logo {{ position:absolute; top:25px; right:40px; font-weight:700; font-size:22px; color:var(--interbank-verde-claro); display:flex; align-items:center; }}
    .header-logo i {{ margin-right:10px; font-size:26px; }}
    .titulo-principal {{ font-weight:700; font-size:32px; color:var(--interbank-azul); margin-bottom:5px; }}
    .subtitulo {{ font-weight:500; font-size:18px; color:var(--interbank-verde-oscuro); margin-bottom:20px; }}
    .triptych-container {{ display:flex; justify-content:space-between; gap:20px; flex:1; height:100%; overflow:hidden; }}
    .panel {{ flex:1; background:white; border-radius:16px; box-shadow:0 4px 15px rgba(0,0,0,.08); display:flex; flex-direction:column; overflow:hidden; border:1px solid #eee; }}
    .panel-header {{ padding:12px; color:white; text-align:center; display:flex; align-items:center; justify-content:center; gap:10px; }}
    .panel-header h2 {{ font-weight:700; font-size:20px; margin:0; letter-spacing:.5px; }}
    .header-auto {{ background-color:var(--interbank-azul); }} .header-manual {{ background-color:var(--interbank-verde-claro); }} .header-semi {{ background-color:var(--interbank-verde-oscuro); }}
    .kpi-grid {{ display:grid; grid-template-columns:1fr 1fr; gap:8px; padding:12px; background-color:var(--bg-gray); }}
    .kpi-card {{ background:white; padding:8px 5px; border-radius:8px; text-align:center; box-shadow:0 2px 4px rgba(0,0,0,.05); }}
    .kpi-card.full-width {{ grid-column:span 2; background-color:var(--interbank-verde-suave); }}
    .kpi-label {{ font-size:14px; color:#555; font-weight:600; margin-bottom:0; text-transform:uppercase; line-height:1.2; }}
    .kpi-value {{ font-size:24px; font-weight:700; color:var(--interbank-azul); line-height:1.2; }}
    .kpi-value.risk {{ color:#d32f2f; }} .kpi-value.success {{ color:var(--interbank-verde-oscuro); }}
    .table-section {{ padding:12px; flex:1; display:flex; flex-direction:column; }}
    .section-title {{ font-size:15px; font-weight:600; color:var(--interbank-verde-oscuro); margin-bottom:8px; border-bottom:2px solid var(--interbank-verde-suave); padding-bottom:4px; }}
    .data-table {{ width:100%; border-collapse:collapse; font-size:14px; }}
    .data-table th {{ text-align:left; padding:6px 4px; color:#444; font-weight:700; background:#f9fafb; border-bottom:1px solid #e5e7eb; white-space:nowrap; }}
    .data-table td {{ padding:6px 4px; border-bottom:1px solid #f0f0f0; color:#333; }}
    .data-table tr:last-child td {{ border-bottom:none; }}
    .group-badge {{ display:inline-block; padding:2px 6px; border-radius:4px; background:#eee; font-weight:600; font-size:13px; color:#333; }}
    .bar-container {{ width:100%; height:8px; background:#eee; border-radius:4px; margin-top:5px; overflow:hidden; }}
    .bar-fill {{ height:100%; border-radius:4px; }}
    .volume-label {{ font-size:14px; color:#555; }}
    .footer-insight {{ margin-top:15px; font-size:15px; color:#444; background:#f0fdf4; padding:12px 20px; border-radius:8px; border-left:5px solid var(--interbank-verde-claro); display:flex; align-items:center; }}
    .footer-insight i {{ color:var(--interbank-verde-claro); margin-right:15px; font-size:20px; }}
    .text-right {{ text-align:right; }} .text-center {{ text-align:center; }} .font-bold {{ font-weight:700; }} .text-red-600 {{ color:#dc2626; }} .text-green-600 {{ color:var(--interbank-verde-oscuro); }}
  </style>
</head>
<body>
  <div class=\"container\">
    <div class=\"header-logo\"><i class=\"fas fa-shield-alt\"></i>INTERBANK</div>
    <h1 class=\"titulo-principal\">ANÁLISIS POR TIPO DE DETECCIÓN</h1>
    <p class=\"subtitulo\">Desempeño y Efectividad por Grupos de Monitoreo - {mes_label}</p>
    <div class=\"triptych-container\">
      {''.join(panels_html)}
    </div>
    <div class=\"footer-insight\"><i class=\"fas fa-lightbulb\"></i><p><strong>Insight:</strong> {insight_txt}</p></div>
  </div>
</body>
</html>
"""

output_dir = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia")
output_file = output_dir / f"2.3.apertura_por_tipologia_{mes_nombre}_{anio}.html"
output_file.write_text(html, encoding='utf-8')
print(f"HTML generado: {output_file}")



HTML generado: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\2.3.apertura_por_tipologia_agosto_2025.html


In [53]:
df_1.head()

,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m
70230,202508,0000081DD593FDADA1F996DE30B1B0323EC1EF690041DA...,19985965,plaft_pj_minorista,20260901,P4,0,0.631293,70213,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
106617,202508,00000E38993C51A1DD6E4048C653C80A2AFB66E34527AB...,20354392,plaft_pj_minorista,20260901,P4,0,0.469248,106600,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
141640,202508,0000BCAB0002120120BC97DA0CF7D372987995159FD0A3...,910935,plaft_pj_minorista,20260901,P4,0,0.235446,141622,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
85596,202508,0000E5A1B73363AEFCAA055EB76D9AE4A7F1C7F46E82D2...,19813872,plaft_pj_minorista,20260901,P4,0,0.565066,85579,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
79641,202508,00015A2F62D6AC753C86D623C0A7BF75BBD6FDF89A909C...,13337711,plaft_pj_minorista,20260901,P4,0,0.593227,79624,NaN,SIN_INFO,NaN,202509,202508,<NA>,0


In [54]:
import numpy as np
import pandas as pd

# =====================================================================================
# Base de envío para NUEVA alerta usando variable1 == 'SIN_INFO'
# Cutoff se CALIBRA en variable1 <> 'SIN_INFO' y se APLICA al universo variable1 == 'SIN_INFO'
# Estrategias en envío: 1) Solo P1  2) Cutoff_score_aplicado
# =====================================================================================

# Parámetros de negocio (ajústalos)
MIN_PRECISION = 0.20   # precisión mínima deseada para elegir cutoff en calibración
TOP_N = None           # opcional: cupo fijo de envío final, ej. 5000

if 'grupo_corte_nueva_alerta' not in df_1.columns:
    raise ValueError("df_1 no tiene la columna grupo_corte_nueva_alerta")
if 'score' not in df_1.columns:
    raise ValueError("df_1 no tiene la columna score")
if 'variable1' not in df_1.columns:
    raise ValueError("df_1 no tiene la columna variable1 necesaria para filtrar SIN_INFO")

# 1) Universos
base_calibracion = df_1[df_1['variable1'].astype(str).str.upper() != 'SIN_INFO'].copy()  # para hallar threshold
base_nueva_alerta = df_1[df_1['variable1'].astype(str).str.upper() == 'SIN_INFO'].copy()  # universo de envío

if base_calibracion.empty:
    raise ValueError("No hay casos para calibrar cutoff (variable1 != 'SIN_INFO')")
if base_nueva_alerta.empty:
    print("[AVISO] No hay casos de envío (variable1 == 'SIN_INFO'). Se continuará con base_envio_final vacía.")

base_calibracion['score_num'] = pd.to_numeric(base_calibracion['score'], errors='coerce')
base_nueva_alerta['score_num'] = pd.to_numeric(base_nueva_alerta['score'], errors='coerce')

base_calibracion = base_calibracion.dropna(subset=['score_num']).copy()
base_nueva_alerta = base_nueva_alerta.dropna(subset=['score_num']).copy()

# 2) Estrategia A (en envío): solo P1
if 'grupo_alerta' in base_nueva_alerta.columns:
    base_envio_p1 = base_nueva_alerta[base_nueva_alerta['grupo_alerta'].astype(str).str.upper() == 'P1'].copy()
else:
    base_envio_p1 = base_nueva_alerta.iloc[0:0].copy()

# 3) Estrategia B: cutoff calibrado en variable1 != SIN_INFO, aplicado al universo SIN_INFO
best_thr = None
best_row = None
metricas_thr = []

tiene_target_calib = 'target_m' in base_calibracion.columns
if tiene_target_calib:
    y_true_calib = pd.to_numeric(base_calibracion['target_m'], errors='coerce').fillna(0).astype(int)

    thr_grid = np.unique(np.quantile(base_calibracion['score_num'], np.linspace(0.50, 0.999, 300)))

    for thr in thr_grid:
        y_pred_calib = (base_calibracion['score_num'] >= thr).astype(int)
        tp = int(((y_true_calib == 1) & (y_pred_calib == 1)).sum())
        fp = int(((y_true_calib == 0) & (y_pred_calib == 1)).sum())
        fn = int(((y_true_calib == 1) & (y_pred_calib == 0)).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        f1 = (2 * precision * recall / (precision + recall)) if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0 else np.nan
        n_sel = int(y_pred_calib.sum())

        metricas_thr.append({
            'threshold': float(thr),
            'n_sel_calibracion': n_sel,
            'tp_calibracion': tp,
            'fp_calibracion': fp,
            'fn_calibracion': fn,
            'precision_calibracion': precision,
            'recall_calibracion': recall,
            'f1_calibracion': f1,
        })

    metricas_thr_df = pd.DataFrame(metricas_thr).sort_values('threshold', ascending=False).reset_index(drop=True)

    candidatos = metricas_thr_df[metricas_thr_df['precision_calibracion'] >= MIN_PRECISION].copy()
    if not candidatos.empty:
        best_row = candidatos.sort_values(
            ['recall_calibracion', 'precision_calibracion', 'f1_calibracion'],
            ascending=False
        ).iloc[0]
    else:
        best_row = metricas_thr_df.sort_values(
            ['f1_calibracion', 'precision_calibracion'],
            ascending=False
        ).iloc[0]

    best_thr = float(best_row['threshold'])
else:
    # Sin target en calibración: fallback no supervisado
    best_thr = float(base_calibracion['score_num'].quantile(0.90))
    metricas_thr_df = pd.DataFrame()

# Aplicar cutoff de calibración al universo de nuevas alertas (variable1 == SIN_INFO)
base_envio_cutoff = base_nueva_alerta[base_nueva_alerta['score_num'] >= best_thr].copy()

if TOP_N is not None and TOP_N > 0:
    base_envio_cutoff = base_envio_cutoff.sort_values('score_num', ascending=False).head(TOP_N).copy()

# 4) Cuadro comparativo final en NUEVAS alertas
resumen_envio = pd.DataFrame({
    'estrategia': ['solo_P1', 'cutoff_score_aplicado'],
    'n_casos': [len(base_envio_p1), len(base_envio_cutoff)],
})

tiene_target_envio = 'target_m' in base_nueva_alerta.columns
if tiene_target_envio:
    total_pos_envio = int(pd.to_numeric(base_nueva_alerta['target_m'], errors='coerce').fillna(0).astype(int).sum())

    base_envio_p1['target_m_num'] = pd.to_numeric(base_envio_p1.get('target_m', 0), errors='coerce').fillna(0).astype(int)
    base_envio_cutoff['target_m_num'] = pd.to_numeric(base_envio_cutoff.get('target_m', 0), errors='coerce').fillna(0).astype(int)

    pos_p1 = int(base_envio_p1['target_m_num'].sum())
    pos_cut = int(base_envio_cutoff['target_m_num'].sum())

    resumen_envio['positivos'] = [pos_p1, pos_cut]
    resumen_envio['precision'] = [
        pos_p1 / len(base_envio_p1) if len(base_envio_p1) > 0 else np.nan,
        pos_cut / len(base_envio_cutoff) if len(base_envio_cutoff) > 0 else np.nan,
    ]
    resumen_envio['recall_sobre_nuevas_alertas'] = [
        pos_p1 / total_pos_envio if total_pos_envio > 0 else np.nan,
        pos_cut / total_pos_envio if total_pos_envio > 0 else np.nan,
    ]

print("=== Universo calibración (variable1 != 'SIN_INFO') ===")
print(f"Casos calibración: {len(base_calibracion):,}")
print("\n=== Universo envío (variable1 == 'SIN_INFO') ===")
print(f"Casos envío: {len(base_nueva_alerta):,}")

print("\n=== Comparación de estrategias de envío ===")
display(resumen_envio)

print("\n=== Cutoff seleccionado (calibrado en variable1 != 'SIN_INFO' y aplicado en variable1 == 'SIN_INFO') ===")
print(f"Threshold score: {best_thr:.6f}")

if tiene_target_calib and best_row is not None:
    print("Métricas del cutoff en calibración (variable1 != 'SIN_INFO'):")
    display(pd.DataFrame([best_row]))

print("\nSugerencia práctica:")
print("- Si priorizas precisión operativa: usar cutoff_score_aplicado.")
print("- Si priorizas simpleza y gobernanza: usar solo_P1.")

# Base final de envío (ajusta aquí la estrategia elegida)
ESTRATEGIA_FINAL = 'cutoff_score_aplicado'   # 'solo_P1' o 'cutoff_score_aplicado'
base_envio_final = base_envio_cutoff.copy() if ESTRATEGIA_FINAL == 'cutoff_score_aplicado' else base_envio_p1.copy()

print(f"\nBase final seleccionada ({ESTRATEGIA_FINAL}): {len(base_envio_final):,} filas")
display(base_envio_final.head())

[AVISO] No hay casos de envío (variable1 == 'SIN_INFO'). Se continuará con base_envio_final vacía.
=== Universo calibración (variable1 != 'SIN_INFO') ===
Casos calibración: 161,415

=== Universo envío (variable1 == 'SIN_INFO') ===
Casos envío: 0

=== Comparación de estrategias de envío ===


,estrategia,n_casos,positivos,precision,recall_sobre_nuevas_alertas
0,solo_P1,0,0,NaN,NaN
1,cutoff_score_aplicado,0,0,NaN,NaN



=== Cutoff seleccionado (calibrado en variable1 != 'SIN_INFO' y aplicado en variable1 == 'SIN_INFO') ===
Threshold score: 0.993046
Métricas del cutoff en calibración (variable1 != 'SIN_INFO'):


,threshold,n_sel_calibracion,tp_calibracion,fp_calibracion,fn_calibracion,precision_calibracion,recall_calibracion,f1_calibracion
0,0.993046,162.0,27.0,135.0,92.0,0.166667,0.226891,0.192171



Sugerencia práctica:
- Si priorizas precisión operativa: usar cutoff_score_aplicado.
- Si priorizas simpleza y gobernanza: usar solo_P1.

Base final seleccionada (cutoff_score_aplicado): 0 filas


,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m,score_num,target_m_num


In [55]:
from pathlib import Path
import pandas as pd

# ================================================================
# HTML NUEVO: Nuevas alertas a compartir (metodología + origen)
# ================================================================
required_vars = ['df_1', 'base_calibracion', 'base_nueva_alerta', 'base_envio_final', 'best_thr', 'ESTRATEGIA_FINAL']
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Faltan variables en memoria: {missing_vars}. Ejecuta primero la celda de estrategia de envío.")

# Periodo
period_value = None
for col in ['codmes_final', 'codmes', 'mes_base']:
    if col in df_1.columns:
        s = pd.to_numeric(df_1[col], errors='coerce').dropna().astype(int).astype(str)
        s = s[s.str.len() >= 6]
        if not s.empty:
            period_value = s.max()[:6]
            break
if period_value is None:
    period_value = '202602'

anio = period_value[:4]
mes_num = period_value[4:6]
meses_es = {
    '01': 'enero', '02': 'febrero', '03': 'marzo', '04': 'abril',
    '05': 'mayo', '06': 'junio', '07': 'julio', '08': 'agosto',
    '09': 'septiembre', '10': 'octubre', '11': 'noviembre', '12': 'diciembre'
}
mes_nombre = meses_es.get(mes_num, f"mes_{mes_num}")
mes_label = f"{mes_nombre.capitalize()} {anio}"

# Métricas principales
total_compartir = int(len(base_envio_final))
total_calibracion = int(len(base_calibracion))
total_nuevas_alertas = int(len(base_nueva_alerta))
threshold = float(best_thr) if pd.notna(best_thr) else float('nan')

# Métricas de calidad en base final (si hay target)
pos_base_final = None
precision_base_final = None
if 'target_m' in base_envio_final.columns and len(base_envio_final) > 0:
    y = pd.to_numeric(base_envio_final['target_m'], errors='coerce').fillna(0).astype(int)
    pos_base_final = int(y.sum())
    precision_base_final = pos_base_final / len(base_envio_final)

# Top variables para transparencia
cols_mostrar = [c for c in ['num_documento', 'codunico', 'grupo_alerta', 'score', 'target_m'] if c in base_envio_final.columns]
preview_df = base_envio_final[cols_mostrar].copy().head(10) if cols_mostrar else pd.DataFrame()

preview_rows = ""
if not preview_df.empty:
    for _, r in preview_df.iterrows():
        vals = [str(r[c]) for c in preview_df.columns]
        preview_rows += "<tr>" + "".join([f"<td>{v}</td>" for v in vals]) + "</tr>"
else:
    preview_rows = "<tr><td colspan='5' class='text-center'>Sin columnas de vista previa</td></tr>"

preview_header = "".join([f"<th>{c}</th>" for c in preview_df.columns]) if not preview_df.empty else "<th>detalle</th>"

precision_txt = f"{precision_base_final*100:.2f}%" if precision_base_final is not None else "N/A"
positivos_txt = str(pos_base_final) if pos_base_final is not None else "N/A"

html = f"""
<!DOCTYPE html>
<html lang=\"es\">
<head>
  <meta charset=\"utf-8\"/>
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\"/>
  <title>Nuevas Alertas a Compartir - {mes_label}</title>
  <link href=\"https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap\" rel=\"stylesheet\"/>
  <link href=\"https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css\" rel=\"stylesheet\"/>
  <link href=\"https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.4.0/css/all.min.css\" rel=\"stylesheet\"/>
  <style>
    :root {{
      --interbank-verde-oscuro: rgb(0,95,30);
      --interbank-verde-claro: rgb(0,190,80);
      --interbank-verde-suave: rgb(198,255,192);
      --interbank-azul: rgb(31,69,146);
      --bg-gray: #f8f9fa;
    }}
    body {{ font-family:'Poppins',sans-serif; background-color:white; margin:0; padding:0; width:1280px; height:720px; overflow:hidden; }}
    .container {{ width:1280px; height:720px; position:relative; padding:20px 30px; display:flex; flex-direction:column; }}
    .header-logo {{ position:absolute; top:20px; right:30px; font-weight:700; font-size:22px; color:var(--interbank-verde-claro); display:flex; align-items:center; }}
    .header-logo i {{ margin-right:10px; font-size:26px; }}
    .titulo-principal {{ font-weight:700; font-size:30px; color:var(--interbank-azul); margin-bottom:4px; }}
    .subtitulo {{ font-weight:500; font-size:17px; color:var(--interbank-verde-oscuro); margin-bottom:12px; }}
    .kpi-grid {{ display:grid; grid-template-columns: repeat(4,1fr); gap:10px; margin-bottom:10px; }}
    .kpi-card {{ background:var(--bg-gray); border:1px solid #eee; border-radius:10px; padding:10px; text-align:center; }}
    .kpi-label {{ font-size:12px; color:#555; font-weight:600; text-transform:uppercase; }}
    .kpi-value {{ font-size:24px; font-weight:700; color:var(--interbank-azul); }}
    .kpi-value.success {{ color:var(--interbank-verde-oscuro); }}
    .kpi-value.risk {{ color:#d32f2f; }}
    .content {{ display:grid; grid-template-columns: 1.05fr 0.95fr; gap:12px; flex:1; min-height:0; }}
    .panel {{ border:1px solid #eee; border-radius:12px; box-shadow:0 4px 15px rgba(0,0,0,.06); overflow:hidden; display:flex; flex-direction:column; }}
    .panel-header {{ background:var(--interbank-azul); color:#fff; font-weight:700; font-size:16px; padding:10px 12px; display:flex; align-items:center; gap:8px; }}
    .panel-body {{ padding:10px 12px; font-size:13px; color:#333; overflow:auto; }}
    .step {{ background:#f9fafb; border-left:4px solid var(--interbank-verde-claro); border-radius:6px; padding:8px 10px; margin-bottom:8px; }}
    .step b {{ color:var(--interbank-verde-oscuro); }}
    .data-table {{ width:100%; border-collapse:collapse; font-size:12px; }}
    .data-table th {{ text-align:left; padding:6px 5px; color:#444; font-weight:700; background:#f9fafb; border-bottom:1px solid #e5e7eb; }}
    .data-table td {{ padding:6px 5px; border-bottom:1px solid #f0f0f0; }}
    .badge {{ display:inline-block; padding:2px 6px; border-radius:4px; background:#eef2ff; color:#1f4592; font-weight:600; }}
    .footer-insight {{ margin-top:10px; font-size:13px; color:#444; background:#f0fdf4; padding:10px 12px; border-radius:8px; border-left:5px solid var(--interbank-verde-claro); }}
  </style>
</head>
<body>
  <div class=\"container\">
    <div class=\"header-logo\"><i class=\"fas fa-shield-alt\"></i>INTERBANK</div>
    <h1 class=\"titulo-principal\">NUEVAS ALERTAS A COMPARTIR</h1>
    <p class=\"subtitulo\">Periodo: {mes_label} · Metodología y trazabilidad del valor final</p>

    <div class=\"kpi-grid\">
      <div class=\"kpi-card\"><div class=\"kpi-label\">Total a Compartir</div><div class=\"kpi-value risk\">{total_compartir}</div></div>
      <div class=\"kpi-card\"><div class=\"kpi-label\">Universo Calibración (variable1 != 'SIN_INFO')</div><div class=\"kpi-value\">{total_calibracion}</div></div>
      <div class=\"kpi-card\"><div class=\"kpi-label\">Universo Envío (variable1 == 'SIN_INFO')</div><div class=\"kpi-value\">{total_nuevas_alertas}</div></div>
      <div class=\"kpi-card\"><div class=\"kpi-label\">Cutoff Score</div><div class=\"kpi-value success\">{threshold:.6f}</div></div>
    </div>

    <div class=\"content\">
      <div class=\"panel\">
        <div class=\"panel-header\"><i class=\"fas fa-list-check\"></i>Cómo se obtuvo el valor</div>
        <div class=\"panel-body\">
          <div class=\"step\"><b>Paso 1:</b> Se arma <span class='badge'>df_1</span> con el cruce de score + target (join por documento/codunico/codmes).</div>
          <div class=\"step\"><b>Paso 2:</b> Se separa calibración en <span class='badge'>variable1 != 'SIN_INFO'</span> y envío en <span class='badge'>variable1 = 'SIN_INFO'</span>.</div>
          <div class=\"step\"><b>Paso 3:</b> El <span class='badge'>cutoff</span> se busca en el universo calibración maximizando recall bajo la restricción de precisión mínima.</div>
          <div class=\"step\"><b>Paso 4:</b> El cutoff hallado se aplica al universo <span class='badge'>variable1 = 'SIN_INFO'</span> para construir <span class='badge'>base_envio_final</span>.</div>
          <div class=\"step\"><b>Paso 5:</b> El total final a compartir es <span class='badge'>len(base_envio_final)</span> = <b>{total_compartir}</b>.</div>

          <h4 style=\"font-weight:700; color:var(--interbank-verde-oscuro); margin:10px 0 6px;\">Origen de datos</h4>
          <ul style=\"margin:0; padding-left:18px; line-height:1.5;\">
            <li>S3 score replica: <span class='badge'>REPLICA_OUTPUT/scr_Plaft-PJ-Minorista_YYYYMM.txt</span></li>
            <li>Athena dataset objetivo: <span class='badge'>d_perm_aws.ds_alertplaft_mdl</span> + <span class='badge'>e_perm_aws.t_alertas_plaft</span></li>
            <li>Notebook de cálculo: <span class='badge'>02.Validacion_final.ipynb</span></li>
          </ul>

          <h4 style=\"font-weight:700; color:var(--interbank-verde-oscuro); margin:10px 0 6px;\">Resultado operativo</h4>
          <p style=\"margin:0;\">Positivos en base final: <b>{positivos_txt}</b> · Precisión observada: <b>{precision_txt}</b></p>
        </div>
      </div>

      <div class=\"panel\">
        <div class=\"panel-header\"><i class=\"fas fa-table\"></i>Vista previa de casos a compartir</div>
        <div class=\"panel-body\">
          <table class=\"data-table\">
            <thead><tr>{preview_header}</tr></thead>
            <tbody>{preview_rows}</tbody>
          </table>
          <div class=\"footer-insight\">
            <strong>Nota:</strong> Este archivo documenta el número final y su trazabilidad para compartir con negocio/auditoría.
          </div>
        </div>
      </div>
    </div>
  </div>
</body>
</html>
"""

out_dir = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia")
out_file = out_dir / f"2.4.nuevas_alertas_compartir_{mes_nombre}_{anio}.html"
out_file.write_text(html, encoding='utf-8')
print(f"HTML nuevo generado: {out_file}")
print(f"Total final a compartir: {total_compartir}")

HTML nuevo generado: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\2.4.nuevas_alertas_compartir_agosto_2025.html
Total final a compartir: 0


In [56]:
variables_importantes = ['codmes','codunico',	'modelo',	'fec_replica',	'grupo_alerta',	'grupo_corte_0.5',	'score',
"ingresos_vs_facturacion",
"trx_monto_abonos_6m_efectivo",
"ciiu_v4",
"edad_constitucion",
"pasivo_vs_ingresos",
"q_alerta_hist",
"sectorista_id",
"q_meses_ingresos_0",
"trx_q_cargos_3m_total",
"ubigeo_cd",
"trx_monto_cargos_6m_efectivo",
"ratio_abonos_1m_vs_6m",
"pasivo_soles",
"antiguedad"
 
]

In [57]:
# Inicializa df_4 para celdas posteriores (head/merge).
# En este notebook, `df_4_temp` ya contiene el universo armado previamente.

if 'df_4_temp' in globals():
    df_4 = df_4_temp.copy()
elif 'df_1' in globals():
    # Fallback si no existe df_4_temp
    df_4 = df_1.copy()
else:
    raise ValueError("No existe df_4_temp ni df_1 en memoria. Ejecuta las celdas anteriores.")

print(f"df_4 listo: {df_4.shape[0]:,} filas, {df_4.shape[1]:,} columnas")

df_4 listo: 161,435 filas, 16 columnas


In [58]:
df_4.head()

,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m
0,202508,8352D6E3CAAA9E1D575EC8D8C87E9CC89F8DFA566B4DA1...,21459911,plaft_pj_minorista,20260901,P1,0,0.997613,1,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
1,202508,70E111221FF5C3F4EE25288AEAE03D08B62444366A2BD0...,21621398,plaft_pj_minorista,20260901,P1,0,0.997542,2,SEMI AUTOMATICA,M,NaN,202509,202508,SEMI AUTOMATICA,1
2,202508,6D436852984D729BF882500D3175566083B44B1AF168B0...,21371474,plaft_pj_minorista,20260901,P1,0,0.997481,3,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
3,202508,824727A560707248E84E3392C179065BF4834D34C9B289...,21662246,plaft_pj_minorista,20260901,P1,0,0.997456,4,NaN,SIN_INFO,NaN,202509,202508,<NA>,0
4,202508,93A1AE6C8FC185B3603CCFDD37FADA60937B357E9748F2...,20281372,plaft_pj_minorista,20260901,P1,0,0.997344,5,NaN,SIN_INFO,NaN,202509,202508,<NA>,0


In [59]:
df_5=base_nueva_alerta

In [60]:
import pandas as pd

# Normaliza tipos para llaves de cruce
if 'codunico' in df_4.columns:
    df_4['codunico'] = pd.to_numeric(df_4['codunico'], errors='coerce').astype('Int64')  # permite NaN
if 'codunico' in df_5.columns:
    df_5['codunico'] = pd.to_numeric(df_5['codunico'], errors='coerce').astype('Int64')
if 'codmes_final' in df_5.columns:
    df_5['codmes_final'] = pd.to_numeric(df_5['codmes_final'], errors='coerce').astype('Int64')  # permite NaN

# Detecta columna de periodo en df_4 (en algunos flujos se llama mes_base)
candidatos_mes_izq = ['mes_base', 'codmes_final', 'codmes', 'codmes_dataset']
mes_izq = next((c for c in candidatos_mes_izq if c in df_4.columns), None)
if mes_izq is None:
    raise ValueError(f"df_4 no tiene columna de periodo. Columnas disponibles: {list(df_4.columns)}")

# Detecta columna de periodo en df_5 (usualmente codmes_final)
candidatos_mes_der = ['codmes_final', 'codmes', 'mes_base']
mes_der = next((c for c in candidatos_mes_der if c in df_5.columns), None)
if mes_der is None:
    raise ValueError(f"df_5 no tiene columna de periodo. Columnas disponibles: {list(df_5.columns)}")

# Valida llaves
keys_izq = ['num_documento', 'codunico', mes_izq]
keys_der = ['num_documento', 'codunico', mes_der]
missing_izq = [c for c in keys_izq if c not in df_4.columns]
missing_der = [c for c in keys_der if c not in df_5.columns]
if missing_izq or missing_der:
    raise ValueError(f"Faltan llaves para merge. missing_izq={missing_izq}, missing_der={missing_der}")

df_6 = pd.merge(
    df_4,
    df_5,
    left_on=keys_izq,
    right_on=keys_der,
    how='inner'
 )

print(f"Merge OK: df_6={df_6.shape}")

Merge OK: df_6=(0, 30)


In [61]:
!pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [41]:
!pip install tabulate

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
from pathlib import Path
import os
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

if 'df_6' not in globals():
    raise ValueError("No existe df_6 en memoria. Ejecuta primero las celdas previas de construcción de base.")

env_path = Path.cwd() / ".env"

def _upsert_env_key(path: Path, key: str, value: str) -> None:
    lines = []
    if path.exists():
        lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()

    updated = False
    new_lines = []
    for line in lines:
        if line.strip().startswith(f"{key}="):
            new_lines.append(f"{key}={value}")
            updated = True
        else:
            new_lines.append(line)

    if not updated:
        if new_lines and new_lines[-1].strip() != "":
            new_lines.append("")
        new_lines.append(f"{key}={value}")

    path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")

def _to_num(v, default=0.0):
    try:
        return float(v)
    except Exception:
        return default

key_from_tu = str(globals().get("TU_API_KEY", "")).strip()
key_from_groq = str(globals().get("GROQ_API_KEY", "")).strip()
key_from_env = os.getenv("GROQ_API_KEY", "").strip()

GROQ_API_KEY = key_from_env or key_from_tu or key_from_groq
USE_LLM = bool(GROQ_API_KEY and GROQ_API_KEY != "TU_API_KEY")

if USE_LLM:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    _upsert_env_key(env_path, "GROQ_API_KEY", GROQ_API_KEY)
    print(f"✓ GROQ_API_KEY sincronizada y guardada en: {env_path}")
else:
    print("⚠ No se encontró TU_API_KEY/GROQ_API_KEY con valor real. Se generarán razones placeholder.")

API_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

ca_bundle = os.getenv("REQUESTS_CA_BUNDLE") or os.getenv("SSL_CERT_FILE")
verify_tls = True
if ca_bundle:
    ca_path = Path(ca_bundle)
    if ca_path.exists():
        verify_tls = str(ca_path)
        print(f"TLS corporativo activo con CA: {ca_path}")
    else:
        print(f"Aviso: REQUESTS_CA_BUNDLE/SSL_CERT_FILE no existe: {ca_bundle}. Se usará CA por defecto.")

session_http = requests.Session()
retry_cfg = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["POST"],
)
session_http.mount("https://", HTTPAdapter(max_retries=retry_cfg))

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json",
}

df_4 = df_6.copy()
columnas = [
    "score",
    "antiguedad",
    "pasivo_soles",
    "trx_monto_cargos_6m_efectivo",
    "trx_q_abonos_ratio_3m_efectivo_total",
    "q_meses_ingresos_0",
]

for col in columnas:
    if col in df_4.columns:
        df_4[col] = pd.to_numeric(df_4[col], errors="coerce")

for col, dec in [
    ("pasivo_soles", 2),
    ("trx_monto_cargos_6m_efectivo", 2),
    ("trx_q_abonos_ratio_3m_efectivo_total", 4),
]:
    if col in df_4.columns:
        df_4[col] = df_4[col].fillna(0).round(dec)

top_100 = df_4.sort_values("score", ascending=False).head(100).copy().reset_index(drop=True)
print("Generando razones con Groq...")

def generar_razon(row):
    if not USE_LLM:
        return "Sin GROQ_API_KEY configurada"

    datos = f"""
Score: {_to_num(row.get('score')):.6f}
Antigüedad: {row.get('antiguedad', 'N/A')}
Pasivo (S/): {_to_num(row.get('pasivo_soles')):,.2f}
Cargos efectivo 6m (S/): {_to_num(row.get('trx_monto_cargos_6m_efectivo')):,.2f}
Ratio abonos 3m: {_to_num(row.get('trx_q_abonos_ratio_3m_efectivo_total')):.4f}
Meses sin ingresos: {row.get('q_meses_ingresos_0', 'N/A')}
"""

    prompt = f"""
Eres un analista experto en prevención de lavado de activos (PLAFT).
Genera una razón sospechosa corta (1-2 líneas) en español y lenguaje simple.

Datos de la cuenta:
{datos}

Responde SOLO con la razón sospechosa.
"""

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 100,
        "temperature": 0.3,
    }

    try:
        response = session_http.post(
            API_URL,
            headers=headers,
            json=payload,
            timeout=45,
            verify=verify_tls,
        )
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"].strip()

        try:
            detail = response.json()
        except Exception:
            detail = response.text[:300]
        return f"Error API {response.status_code}: {detail}"
    except requests.exceptions.SSLError as e:
        return f"Error SSL: {e}. Configura REQUESTS_CA_BUNDLE con el CA corporativo."
    except requests.exceptions.RequestException as e:
        return f"Error conexión: {e}"

razones = []
for i, row in top_100.iterrows():
    print(f"  → {i+1}/100 | Score: {_to_num(row.get('score')):.6f}")
    razones.append(generar_razon(row))
    time.sleep(0.5)

top_100["razon_sospechosa"] = razones
df_4["razon_sospechosa"] = ""
df_4 = df_4.sort_values("score", ascending=False).reset_index(drop=True)
df_4.loc[0:99, "razon_sospechosa"] = razones

print("\nListo: razones generadas para top 100.")
print(
    top_100[
        [
            "score",
            "pasivo_soles",
            "trx_monto_cargos_6m_efectivo",
            "trx_q_abonos_ratio_3m_efectivo_total",
            "q_meses_ingresos_0",
            "razon_sospechosa",
        ]
    ].head(10).to_string(index=False)
)

⚠ No se encontró TU_API_KEY/GROQ_API_KEY con valor real. Se generarán razones placeholder.


KeyError: 'score'

In [ ]:
# No hardcodear claves: usar variable de entorno GROQ_API_KEY (o un archivo .env)
TU_API_KEY = os.getenv("GROQ_API_KEY", "")

In [44]:
from pathlib import Path
print("CWD:", Path.cwd())
print("env_path:", env_path if 'env_path' in globals() else "<no definido>")
if 'env_path' in globals():
    print("env exists:", Path(env_path).exists())

CWD: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia
env_path: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\.env
env exists: False


In [45]:
# Resumen por grupo_alerta: total de casos y casos positivos (target_m = 1)
if 'target_m' not in df_3.columns:
    raise ValueError("df_3 no tiene la columna target_m")

resumen_grupo = (
    df_3.assign(target_m_num=pd.to_numeric(df_3['target_m'], errors='coerce').fillna(0).astype(int))
       .groupby('grupo_alerta', as_index=False)
       .agg(
           total_casos=('target_m_num', 'size'),
           casos_positivos=('target_m_num', 'sum')
       )
       .sort_values('grupo_alerta')
)

resumen_grupo['tasa_positivos'] = (resumen_grupo['casos_positivos'] / resumen_grupo['total_casos']).round(4)

print("=== Casos por grupo_alerta ===")
display(resumen_grupo)

=== Casos por grupo_alerta ===


,grupo_alerta,total_casos,casos_positivos,tasa_positivos
0,P1,156,35,0.2244
1,P2,255,26,0.1020
2,P3,127,12,0.0945
3,P4+P5,120,3,0.0250


In [46]:
# Resumen por grupo_alerta: total de casos y casos positivos (target_m = 1)
if 'target_m' not in df_1.columns:
    raise ValueError("df_2 no tiene la columna target_m")

resumen_grupo = (
    df_1.assign(target_m_num=pd.to_numeric(df_1['target_m'], errors='coerce').fillna(0).astype(int))
       .groupby('grupo_alerta', as_index=False)
       .agg(
           total_casos=('target_m_num', 'size'),
           casos_positivos=('target_m_num', 'sum')
       )
       .sort_values('grupo_alerta')
)

resumen_grupo['tasa_positivos'] = (resumen_grupo['casos_positivos'] / resumen_grupo['total_casos']).round(4)

print("=== Casos por grupo_alerta ===")
display(resumen_grupo)

=== Casos por grupo_alerta ===


,grupo_alerta,total_casos,casos_positivos,tasa_positivos
0,P1,156,35,0.2244
1,P2,255,26,0.1020
2,P3,127,12,0.0945
3,P4+P5,236089,3,0.0000


In [47]:
from pathlib import Path
import pandas as pd

# ====================================================================
# HTML NUEVO: Comparativo por grupo (sin filtro vs con filtro df_3) + alertas a generar
# + Apertura adicional del universo sin filtro en grupos de 1,000 (top 10 grupos)
# ====================================================================
if 'df_1' not in globals():
    raise ValueError("No existe df_1 en memoria.")

required_cols = {'grupo_alerta', 'target_m'}
missing = required_cols.difference(df_1.columns)
if missing:
    raise ValueError(f"Faltan columnas requeridas en df_1: {missing}")

# Universos
base_sin_filtro = df_1.copy()

# El filtro solicitado por usuario: exactamente el usado en el cuadro de df_3
if 'df_3' in globals() and isinstance(df_3, pd.DataFrame):
    base_con_filtro = df_3.copy()
else:
    base_con_filtro = df_1[df_1['variable1'].astype(str) != 'SIN_INFO'].copy() if 'variable1' in df_1.columns else pd.DataFrame(columns=df_1.columns)

if 'base_envio_final' in globals() and isinstance(base_envio_final, pd.DataFrame):
    base_generar = base_envio_final.copy()
    estrategia_lbl = str(globals().get('ESTRATEGIA_FINAL', 'No definida'))
else:
    base_generar = base_con_filtro.copy()
    estrategia_lbl = 'No definida (usa base_con_filtro)'

def _prep_target(df: pd.DataFrame) -> pd.DataFrame:
    tmp = df.copy()
    tmp['target_m_num'] = pd.to_numeric(tmp['target_m'], errors='coerce').fillna(0).astype(int)
    return tmp

def _resumen_target(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['grupo_alerta', 'total_casos', 'casos_positivos', 'tasa_positivos'])
    r = (
        _prep_target(df)
        .groupby('grupo_alerta', as_index=False)
        .agg(total_casos=('target_m_num', 'size'), casos_positivos=('target_m_num', 'sum'))
        .sort_values('grupo_alerta')
    )
    r['tasa_positivos'] = (r['casos_positivos'] / r['total_casos']).fillna(0)
    return r

def _prep_previas(df: pd.DataFrame) -> pd.DataFrame:
    tmp = df.copy()
    if 'q_alerta_hist' in tmp.columns:
        tmp['alertas_previas_num'] = pd.to_numeric(tmp['q_alerta_hist'], errors='coerce').fillna(0)
    elif 'grupo_corte_nueva_alerta' in tmp.columns:
        tmp['alertas_previas_num'] = (pd.to_numeric(tmp['grupo_corte_nueva_alerta'], errors='coerce').fillna(1) == 0).astype(int)
    else:
        tmp['alertas_previas_num'] = 0
    return tmp

def _resumen_previas(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['grupo_alerta', 'total_casos', 'alertas_previas', 'pct_sin_previas'])
    r = (
        _prep_previas(df)
        .groupby('grupo_alerta', as_index=False)
        .agg(total_casos=('alertas_previas_num', 'size'), alertas_previas=('alertas_previas_num', 'sum'))
        .sort_values('grupo_alerta')
    )
    r['pct_sin_previas'] = ((r['total_casos'] - r['alertas_previas']) / r['total_casos']).fillna(0)
    return r

res_sin = _resumen_target(base_sin_filtro)
res_con = _resumen_target(base_con_filtro)
res_gen = _resumen_previas(base_generar)

def fmt_pct(x: float) -> str:
    return f"{x*100:.2f}%"

def table_rows_target(df_res: pd.DataFrame) -> str:
    if df_res.empty:
        return "<tr><td colspan='4' class='text-center'>Sin datos</td></tr>"
    rows = []
    for _, r in df_res.iterrows():
        riesgo_cls = "text-center text-red-600 font-bold" if int(r['casos_positivos']) > 0 else "text-center"
        efect_cls = "text-right font-bold text-green-600" if float(r['tasa_positivos']) >= 0.5 else "text-right font-bold"
        rows.append(
            f"<tr>"
            f"<td><span class='group-badge'>{r['grupo_alerta']}</span></td>"
            f"<td class='text-center'>{int(r['total_casos'])}</td>"
            f"<td class='{riesgo_cls}'>{int(r['casos_positivos'])}</td>"
            f"<td class='{efect_cls}'>{fmt_pct(float(r['tasa_positivos']))}</td>"
            f"</tr>"
        )
    return ''.join(rows)

def table_rows_previas(df_res: pd.DataFrame) -> str:
    if df_res.empty:
        return "<tr><td colspan='4' class='text-center'>Sin datos</td></tr>"
    rows = []
    for _, r in df_res.iterrows():
        previas_cls = "text-center text-red-600 font-bold" if float(r['alertas_previas']) > 0 else "text-center text-green-600 font-bold"
        pct_cls = "text-right font-bold text-green-600" if float(r['pct_sin_previas']) >= 0.8 else "text-right font-bold"
        rows.append(
            f"<tr>"
            f"<td><span class='group-badge'>{r['grupo_alerta']}</span></td>"
            f"<td class='text-center'>{int(r['total_casos'])}</td>"
            f"<td class='{previas_cls}'>{float(r['alertas_previas']):.0f}</td>"
            f"<td class='{pct_cls}'>{fmt_pct(float(r['pct_sin_previas']))}</td>"
            f"</tr>"
        )
    return ''.join(rows)

sin_total = int(len(base_sin_filtro))
con_total = int(len(base_con_filtro))
gen_total = int(len(base_generar))

sin_pos = int(pd.to_numeric(base_sin_filtro['target_m'], errors='coerce').fillna(0).astype(int).sum()) if not base_sin_filtro.empty else 0
con_pos = int(pd.to_numeric(base_con_filtro['target_m'], errors='coerce').fillna(0).astype(int).sum()) if not base_con_filtro.empty else 0

if not base_generar.empty:
    tmp_prev = _prep_previas(base_generar)
    gen_previas = float(tmp_prev['alertas_previas_num'].sum())
else:
    gen_previas = 0.0

sin_ef = (sin_pos / sin_total) if sin_total > 0 else 0.0
con_ef = (con_pos / con_total) if con_total > 0 else 0.0
gen_sin_previas = ((gen_total - gen_previas) / gen_total) if gen_total > 0 else 0.0

# Apertura adicional: grupos de 1,000 sobre el cuadro 1 (sin filtro), top 10 grupos
if 'score' in base_sin_filtro.columns and not base_sin_filtro.empty:
    base_1000 = base_sin_filtro.copy()
    base_1000['score_num'] = pd.to_numeric(base_1000['score'], errors='coerce')
    base_1000['target_m_num'] = pd.to_numeric(base_1000['target_m'], errors='coerce').fillna(0).astype(int)
    base_1000 = base_1000.sort_values('score_num', ascending=False, na_position='last').reset_index(drop=True)
    base_1000['grupo_1000'] = (base_1000.index // 1000) + 1

    apertura_1000 = (
        base_1000.groupby('grupo_1000', as_index=False)
        .agg(total_casos=('target_m_num', 'size'), casos_positivos=('target_m_num', 'sum'))
    )
    apertura_1000['tasa_positivos'] = (apertura_1000['casos_positivos'] / apertura_1000['total_casos']).fillna(0)
    apertura_1000 = apertura_1000.sort_values('grupo_1000').head(10)

    rows_1000 = []
    for _, r in apertura_1000.iterrows():
        rows_1000.append(
            f"<tr>"
            f"<td class='text-center'>G{int(r['grupo_1000'])}</td>"
            f"<td class='text-center'>{int(r['total_casos'])}</td>"
            f"<td class='text-center text-red-600 font-bold'>{int(r['casos_positivos'])}</td>"
            f"<td class='text-right font-bold'>{fmt_pct(float(r['tasa_positivos']))}</td>"
            f"</tr>"
        )
    rows_1000_html = ''.join(rows_1000) if rows_1000 else "<tr><td colspan='4' class='text-center'>Sin datos</td></tr>"
else:
    rows_1000_html = "<tr><td colspan='4' class='text-center'>Sin columna score para calcular grupos de 1,000</td></tr>"

# Periodo
period_value = None
for col in ['codmes_final', 'codmes', 'mes_base']:
    if col in df_1.columns:
        s = pd.to_numeric(df_1[col], errors='coerce').dropna().astype(int).astype(str)
        s = s[s.str.len() >= 6]
        if not s.empty:
            period_value = s.max()[:6]
            break
if period_value is None:
    period_value = '202602'

anio = period_value[:4]
mes_num = period_value[4:6]
meses_es = {
    '01': 'enero', '02': 'febrero', '03': 'marzo', '04': 'abril',
    '05': 'mayo', '06': 'junio', '07': 'julio', '08': 'agosto',
    '09': 'septiembre', '10': 'octubre', '11': 'noviembre', '12': 'diciembre'
}
mes_nombre = meses_es.get(mes_num, f"mes_{mes_num}")
mes_label = f"{mes_nombre.capitalize()} {anio}"

html = f"""
<!DOCTYPE html>
<html lang=\"es\">
<head>
  <meta charset=\"utf-8\"/>
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\"/>
  <title>Comparativo por Grupo - Interbank</title>
  <link href=\"https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap\" rel=\"stylesheet\"/>
  <link href=\"https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css\" rel=\"stylesheet\"/>
  <link href=\"https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.4.0/css/all.min.css\" rel=\"stylesheet\"/>
  <style>
    :root {{ --interbank-verde-oscuro: rgb(0,95,30); --interbank-verde-claro: rgb(0,190,80); --interbank-verde-suave: rgb(198,255,192); --interbank-azul: rgb(31,69,146); --bg-gray:#f8f9fa; }}
    body {{ font-family:'Poppins',sans-serif; background-color:white; margin:0; padding:0; width:1280px; height:720px; overflow:hidden; }}
    .container {{ width:1280px; height:720px; position:relative; padding:18px 34px; display:flex; flex-direction:column; }}
    .header-logo {{ position:absolute; top:22px; right:34px; font-weight:700; font-size:22px; color:var(--interbank-verde-claro); display:flex; align-items:center; }}
    .header-logo i {{ margin-right:10px; font-size:26px; }}
    .titulo-principal {{ font-weight:700; font-size:28px; color:var(--interbank-azul); margin-bottom:4px; }}
    .subtitulo {{ font-weight:500; font-size:16px; color:var(--interbank-verde-oscuro); margin-bottom:12px; }}
    .triptych-container {{ display:flex; justify-content:space-between; gap:14px; flex:1; min-height:0; overflow:hidden; }}
    .panel {{ flex:1; background:white; border-radius:14px; box-shadow:0 4px 15px rgba(0,0,0,.08); display:flex; flex-direction:column; overflow:hidden; border:1px solid #eee; }}
    .panel-header {{ padding:10px; color:white; text-align:center; display:flex; align-items:center; justify-content:center; gap:8px; }}
    .panel-header h2 {{ font-weight:700; font-size:17px; margin:0; letter-spacing:.3px; }}
    .header-a {{ background-color:var(--interbank-azul); }} .header-b {{ background-color:var(--interbank-verde-claro); }} .header-c {{ background-color:var(--interbank-verde-oscuro); }}
    .kpi-grid {{ display:grid; grid-template-columns:1fr 1fr; gap:6px; padding:10px; background-color:var(--bg-gray); }}
    .kpi-card {{ background:white; padding:7px 5px; border-radius:8px; text-align:center; box-shadow:0 2px 4px rgba(0,0,0,.05); }}
    .kpi-card.full-width {{ grid-column:span 2; background-color:var(--interbank-verde-suave); }}
    .kpi-label {{ font-size:12px; color:#555; font-weight:600; margin-bottom:0; text-transform:uppercase; line-height:1.2; }}
    .kpi-value {{ font-size:21px; font-weight:700; color:var(--interbank-azul); line-height:1.2; }}
    .kpi-value.risk {{ color:#d32f2f; }} .kpi-value.success {{ color:var(--interbank-verde-oscuro); }}
    .table-section {{ padding:10px; flex:1; display:flex; flex-direction:column; min-height:0; }}
    .section-title {{ font-size:14px; font-weight:600; color:var(--interbank-verde-oscuro); margin-bottom:6px; border-bottom:2px solid var(--interbank-verde-suave); padding-bottom:3px; }}
    .data-table {{ width:100%; border-collapse:collapse; font-size:12px; }}
    .data-table th {{ text-align:left; padding:5px 4px; color:#444; font-weight:700; background:#f9fafb; border-bottom:1px solid #e5e7eb; white-space:nowrap; }}
    .data-table td {{ padding:5px 4px; border-bottom:1px solid #f0f0f0; color:#333; }}
    .group-badge {{ display:inline-block; padding:2px 6px; border-radius:4px; background:#eee; font-weight:600; font-size:11px; color:#333; }}
    .footer-insight {{ margin-top:8px; font-size:12px; color:#444; background:#f0fdf4; padding:8px 10px; border-radius:8px; border-left:5px solid var(--interbank-verde-claro); }}
    .extra-box {{ margin-top:10px; background:white; border-radius:14px; box-shadow:0 4px 15px rgba(0,0,0,.08); border:1px solid #eee; overflow:hidden; }}
    .extra-header {{ background:var(--interbank-azul); color:white; font-weight:700; padding:8px 12px; font-size:14px; }}
    .extra-body {{ padding:8px 10px; max-height:140px; overflow:auto; }}
    .text-right {{ text-align:right; }} .text-center {{ text-align:center; }} .font-bold {{ font-weight:700; }} .text-red-600 {{ color:#dc2626; }} .text-green-600 {{ color:var(--interbank-verde-oscuro); }}
  </style>
</head>
<body>
  <div class=\"container\">
    <div class=\"header-logo\"><i class=\"fas fa-shield-alt\"></i>INTERBANK</div>
    <h1 class=\"titulo-principal\">MÉTRICAS POR GRUPO ALERTA</h1>
    <p class=\"subtitulo\">Comparativo sin filtro vs filtro de negocio (df_3) + alertas a generar · {mes_label}</p>

    <div class=\"triptych-container\">
      <div class=\"panel\">
        <div class=\"panel-header header-a\"><i class=\"fas fa-layer-group\"></i><h2>SIN FILTRO</h2></div>
        <div class=\"kpi-grid\">
          <div class=\"kpi-card\"><p class=\"kpi-label\">Total casos</p><p class=\"kpi-value\">{sin_total}</p></div>
          <div class=\"kpi-card\"><p class=\"kpi-label\">Positivos</p><p class=\"kpi-value risk\">{sin_pos}</p></div>
          <div class=\"kpi-card full-width\"><p class=\"kpi-label\">Efectividad global</p><p class=\"kpi-value success\">{fmt_pct(sin_ef)}</p></div>
        </div>
        <div class=\"table-section\">
          <h3 class=\"section-title\">Por grupo_alerta</h3>
          <table class=\"data-table\">
            <thead><tr><th>Grupo</th><th class=\"text-center\">Total</th><th class=\"text-center\">Positivos</th><th class=\"text-right\">Efect.</th></tr></thead>
            <tbody>{table_rows_target(res_sin)}</tbody>
          </table>
        </div>
      </div>

      <div class=\"panel\">
        <div class=\"panel-header header-b\"><i class=\"fas fa-filter\"></i><h2>CON FILTRO (df_3)</h2></div>
        <div class=\"kpi-grid\">
          <div class=\"kpi-card\"><p class=\"kpi-label\">Total casos</p><p class=\"kpi-value\">{con_total}</p></div>
          <div class=\"kpi-card\"><p class=\"kpi-label\">Positivos</p><p class=\"kpi-value risk\">{con_pos}</p></div>
          <div class=\"kpi-card full-width\"><p class=\"kpi-label\">Efectividad global</p><p class=\"kpi-value success\">{fmt_pct(con_ef)}</p></div>
        </div>
        <div class=\"table-section\">
          <h3 class=\"section-title\">Por grupo_alerta</h3>
          <table class=\"data-table\">
            <thead><tr><th>Grupo</th><th class=\"text-center\">Total</th><th class=\"text-center\">Positivos</th><th class=\"text-right\">Efect.</th></tr></thead>
            <tbody>{table_rows_target(res_con)}</tbody>
          </table>
        </div>
      </div>

      <div class=\"panel\">
        <div class=\"panel-header header-c\"><i class=\"fas fa-paper-plane\"></i><h2>ALERTAS A GENERAR</h2></div>
        <div class=\"kpi-grid\">
          <div class=\"kpi-card\"><p class=\"kpi-label\">Total a generar</p><p class=\"kpi-value\">{gen_total}</p></div>
          <div class=\"kpi-card\"><p class=\"kpi-label\">Alertas previas</p><p class=\"kpi-value risk\">{gen_previas:.0f}</p></div>
          <div class=\"kpi-card full-width\"><p class=\"kpi-label\">% sin alertas previas</p><p class=\"kpi-value success\">{fmt_pct(gen_sin_previas)}</p></div>
        </div>
        <div class=\"table-section\">
          <h3 class=\"section-title\">Por grupo_alerta</h3>
          <table class=\"data-table\">
            <thead><tr><th>Grupo</th><th class=\"text-center\">Total</th><th class=\"text-center\">Alertas previas</th><th class=\"text-right\">% sin previas</th></tr></thead>
            <tbody>{table_rows_previas(res_gen)}</tbody>
          </table>
          <div class=\"footer-insight\"><strong>Estrategia:</strong> {estrategia_lbl}</div>
        </div>
      </div>
    </div>

    <div class=\"extra-box\">
      <div class=\"extra-header\"><i class=\"fas fa-list-ol\"></i> Apertura del cuadro 1 en grupos de 1,000 (primeros 10 grupos)</div>
      <div class=\"extra-body\">
        <table class=\"data-table\">
          <thead><tr><th class=\"text-center\">Grupo 1,000</th><th class=\"text-center\">Total</th><th class=\"text-center\">Positivos</th><th class=\"text-right\">Efect.</th></tr></thead>
          <tbody>{rows_1000_html}</tbody>
        </table>
      </div>
    </div>
  </div>
</body>
</html>
"""

output_dir = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia")
output_file = output_dir / f"2.5.comparativo_grupo_alerta_{mes_nombre}_{anio}.html"
output_file.write_text(html, encoding='utf-8')
print(f"HTML generado: {output_file}")

HTML generado: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\2.5.comparativo_grupo_alerta_febrero_2026.html


In [48]:
from pathlib import Path
import pandas as pd

# Ajuste del 4to cuadro: mostrar EXACTAMENTE 1,000 grupos (por ranking de score) y sin recorte visual
if 'df_1' not in globals():
    raise ValueError("No existe df_1 en memoria.")

N_GRUPOS = 1000

period_value = None
for col in ['codmes_final', 'codmes', 'mes_base']:
    if col in df_1.columns:
        s = pd.to_numeric(df_1[col], errors='coerce').dropna().astype(int).astype(str)
        s = s[s.str.len() >= 6]
        if not s.empty:
            period_value = s.max()[:6]
            break
if period_value is None:
    period_value = '202602'

anio = period_value[:4]
mes_num = period_value[4:6]
meses_es = {
    '01': 'enero', '02': 'febrero', '03': 'marzo', '04': 'abril',
    '05': 'mayo', '06': 'junio', '07': 'julio', '08': 'agosto',
    '09': 'septiembre', '10': 'octubre', '11': 'noviembre', '12': 'diciembre'
}
mes_nombre = meses_es.get(mes_num, f"mes_{mes_num}")

html_path = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/notebook/inferencia") / f"2.5.comparativo_grupo_alerta_{mes_nombre}_{anio}.html"
if not html_path.exists():
    raise FileNotFoundError(f"No existe el HTML esperado: {html_path}")

base_rank = df_1.copy()
if 'score' not in base_rank.columns or 'target_m' not in base_rank.columns:
    raise ValueError("df_1 debe contener 'score' y 'target_m' para calcular la apertura")

base_rank['score_num'] = pd.to_numeric(base_rank['score'], errors='coerce')
base_rank['target_m_num'] = pd.to_numeric(base_rank['target_m'], errors='coerce').fillna(0).astype(int)
base_rank = base_rank.sort_values('score_num', ascending=False, na_position='last').reset_index(drop=True)

n = len(base_rank)
if n == 0:
    raise ValueError("df_1 está vacío; no se puede construir la apertura de grupos")

# 1,000 grupos exactos por ranking (si n >= 1000)
base_rank['grupo_1000'] = ((base_rank.index * N_GRUPOS) // n) + 1
base_rank['grupo_1000'] = base_rank['grupo_1000'].clip(upper=N_GRUPOS)

apertura_1000 = (
    base_rank.groupby('grupo_1000', as_index=False)
    .agg(total_casos=('target_m_num', 'size'), casos_positivos=('target_m_num', 'sum'))
    .sort_values('grupo_1000')
    .reset_index(drop=True)
)
apertura_1000['tasa_positivos'] = (apertura_1000['casos_positivos'] / apertura_1000['total_casos']).fillna(0)

rows_1000_html = ''.join(
    [
        f"<tr><td class='text-center'>G{int(r.grupo_1000)}</td><td class='text-center'>{int(r.total_casos)}</td><td class='text-center text-red-600 font-bold'>{int(r.casos_positivos)}</td><td class='text-right font-bold'>{r.tasa_positivos*100:.2f}%</td></tr>"
        for _, r in apertura_1000.iterrows()
    ]
)
if not rows_1000_html:
    rows_1000_html = "<tr><td colspan='4' class='text-center'>Sin datos</td></tr>"

html = html_path.read_text(encoding='utf-8')

html = html.replace("Apertura del cuadro 1 en grupos de 1,000 (todos los grupos)", "Apertura del cuadro 1 en 1,000 grupos (todos los grupos)")
html = html.replace("Apertura del cuadro 1 en grupos de 1,000 (primeros 10 grupos)", "Apertura del cuadro 1 en 1,000 grupos (todos los grupos)")
html = html.replace("width:1280px; height:720px; overflow:hidden;", "width:1280px; min-height:720px; overflow:auto;")
html = html.replace("width:1280px; height:720px;", "width:1280px; min-height:720px;")
html = html.replace("max-height:140px; overflow:auto;", "overflow:visible;")
html = html.replace("<th class=\"text-center\">Grupo 1,000</th>", "<th class=\"text-center\">Grupo (1..1000)</th>")

start_marker = '<div class="extra-box">'
thead_marker = '<thead><tr><th class="text-center">Grupo'
tbody_open = '<tbody>'
tbody_close = '</tbody>'

start_idx = html.find(start_marker)
if start_idx == -1:
    raise ValueError("No se encontró el bloque extra-box en el HTML")

thead_idx = html.find(thead_marker, start_idx)
if thead_idx == -1:
    raise ValueError("No se encontró la cabecera del cuadro extra")

tbody_start_idx = html.find(tbody_open, thead_idx)
tbody_end_idx = html.find(tbody_close, tbody_start_idx)
if tbody_start_idx == -1 or tbody_end_idx == -1:
    raise ValueError("No se encontró el tbody del cuadro extra")

before = html[:tbody_start_idx + len(tbody_open)]
after = html[tbody_end_idx:]
html = before + rows_1000_html + after

html_path.write_text(html, encoding='utf-8')
print(f"HTML actualizado: {html_path}")
print(f"Total grupos mostrados: {len(apertura_1000)} de {N_GRUPOS} solicitados")
print(f"Total registros distribuidos: {n}")

ValueError: No se encontró la cabecera del cuadro extra

In [10]:
from pathlib import Path

ruta_salida = Path("inferencia_masivo_0926.csv")
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
print(f"CSV guardado: {ruta_salida.resolve()} | Filas: {len(df):,}")

CSV guardado: C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Minorista_regulado\Inferencia\inferencia_masivo_0926.csv | Filas: 180,309


In [11]:
# Diagnostico de caida P1: 202508 (referencia) vs 202609
COLS_VARS_DIAGNOSTICO = [
    "cnt_trx_cargostot_3m", "mto_pas_soles", "rat_pastot_x_ingtot_6m",
    "cnt_trx_abonospromtot_3m", "imp_trx_abonosefect_6m", "num_antiguedad",
    "imp_trx_cargosefe_6m", "ratio_cargos_1m_vs_6m", "cnt_meses_sinegresos_12m",
    "cnt_alerta_hist", "rat_trx_abonosefectot_3m", "avg_trx_cargostot_3m",
    "rat_mntcrgsefetot_1m", "rat_trx_abonosefectot_1m", "rat_cntros_x_cnttrxegr_3m",
    "share_cp_egresos", "mto_del_ext_12m", "rat_ing_ext_x_ing_tot_12m",
    "max_mto_cpegrmen_12m", "avg_cpmenegr_12m", "mto_al_ext_12m",
    "ratio_egresos_exterior", "cnt_ros_hist", "cnt_trx_sinenv_alext_12m",
    "flg_alerta_12m", "flg_vrcn_abonos_5m_1m", "avg_cp_men_ing_12m",
    "cnt_noticias", "share_cp_ingresos", "mto_fact_declarado_sunat",
    "cod_ubigeo_cd", "cod_sectorista_id",
]

ruta_202508 = (
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/"
    "scr_PLAFPJMINORISTA_202508.txt"
)
df_202508 = s3_read_csv(ruta_202508, sep="|")
df_202609 = df.copy()

for data in (df_202508, df_202609):
    data.columns = data.columns.str.strip()
    data["grupo_alerta"] = data["grupo_alerta"].astype(str).str.upper().str.strip()

resumen_p1 = pd.DataFrame({
    "periodo": ["202508", "202609"],
    "universo": [len(df_202508), len(df_202609)],
    "P1": [(df_202508["grupo_alerta"] == "P1").sum(), (df_202609["grupo_alerta"] == "P1").sum()],
})
resumen_p1["pct_P1"] = resumen_p1["P1"] / resumen_p1["universo"] * 100
print("Resumen de P1")
display(resumen_p1)

# PSI calcula el cambio de distribución de cada variable usando deciles de 202508.
def calcular_psi(referencia, actual, bins=10):
    referencia = pd.to_numeric(referencia, errors="coerce").dropna()
    actual = pd.to_numeric(actual, errors="coerce").dropna()
    if referencia.empty or actual.empty or referencia.nunique() < 2:
        return np.nan

    cortes = np.unique(np.nanquantile(referencia, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return np.nan
    cortes[0], cortes[-1] = -np.inf, np.inf
    ref_pct = pd.cut(referencia, bins=cortes, include_lowest=True).value_counts(normalize=True, sort=False)
    act_pct = pd.cut(actual, bins=cortes, include_lowest=True).value_counts(normalize=True, sort=False)
    ref_pct = ref_pct.clip(lower=0.0001)
    act_pct = act_pct.reindex(ref_pct.index, fill_value=0.0001).clip(lower=0.0001)
    return ((act_pct - ref_pct) * np.log(act_pct / ref_pct)).sum()

filas_drift = []
for variable in COLS_VARS_DIAGNOSTICO:
    disponible_202508 = variable in df_202508.columns
    disponible_202609 = variable in df_202609.columns
    fila = {
        "variable": variable,
        "estado": "OK" if disponible_202508 and disponible_202609 else "No disponible en ambos periodos",
        "psi": np.nan,
        "na_202508_pct": np.nan,
        "na_202609_pct": np.nan,
        "mediana_202508": np.nan,
        "mediana_202609": np.nan,
        "media_202508": np.nan,
        "media_202609": np.nan,
    }
    if disponible_202508 and disponible_202609:
        ref = pd.to_numeric(df_202508[variable], errors="coerce")
        act = pd.to_numeric(df_202609[variable], errors="coerce")
        fila.update({
            "psi": calcular_psi(ref, act),
            "na_202508_pct": ref.isna().mean() * 100,
            "na_202609_pct": act.isna().mean() * 100,
            "mediana_202508": ref.median(),
            "mediana_202609": act.median(),
            "media_202508": ref.mean(),
            "media_202609": act.mean(),
        })
    filas_drift.append(fila)

df_drift_variables = pd.DataFrame(filas_drift).sort_values("psi", ascending=False, na_position="last")
variables_faltantes_202508 = [v for v in COLS_VARS_DIAGNOSTICO if v not in df_202508.columns]
variables_faltantes_202609 = [v for v in COLS_VARS_DIAGNOSTICO if v not in df_202609.columns]

print("Variables con mayor cambio de distribución: PSI > 0.25 alto; 0.10-0.25 moderado")
display(df_drift_variables)
print("Ausentes en 202508:", variables_faltantes_202508 or "ninguna")
print("Ausentes en 202609:", variables_faltantes_202609 or "ninguna")

df_drift_variables.head(10)

KeyError: 'grupo_alerta'

In [17]:
# Drift de variables: datasets de entrada SageMaker, no REPLICA_OUTPUT
PERIODO_REF = "202508"
PERIODO_ACTUAL = "202609"

def calcular_psi(referencia, actual, bins=10):
    referencia = pd.to_numeric(referencia, errors="coerce").dropna()
    actual = pd.to_numeric(actual, errors="coerce").dropna()
    if referencia.empty or actual.empty or referencia.nunique() < 2:
        return np.nan

    cortes = np.unique(np.nanquantile(referencia, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return np.nan

    cortes[0], cortes[-1] = -np.inf, np.inf
    ref_pct = pd.cut(referencia, bins=cortes, include_lowest=True).value_counts(normalize=True, sort=False)
    act_pct = pd.cut(actual, bins=cortes, include_lowest=True).value_counts(normalize=True, sort=False)
    ref_pct = ref_pct.clip(lower=0.0001)
    act_pct = act_pct.reindex(ref_pct.index, fill_value=0.0001).clip(lower=0.0001)
    return ((act_pct - ref_pct) * np.log(act_pct / ref_pct)).sum()

ruta_input_ref = f"s3://{bucket_name}/{model_prefix}/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo={PERIODO_REF}/"
ruta_input_actual = f"s3://{bucket_name}/{model_prefix}/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo={PERIODO_ACTUAL}/"

df_input_202508 = wr.s3.read_parquet(ruta_input_ref, boto3_session=session)
df_input_202609 = wr.s3.read_parquet(ruta_input_actual, boto3_session=session)
df_input_202508.columns = df_input_202508.columns.str.strip()
df_input_202609.columns = df_input_202609.columns.str.strip()

# El P1 se obtiene desde REPLICA_OUTPUT; las variables se comparan desde INPUT.
ruta_score_202508 = f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_PLAFPJMINORISTA_{PERIODO_REF}.txt"
df_score_202508 = s3_read_csv(ruta_score_202508, sep="|")
df_score_202508.columns = df_score_202508.columns.str.strip()
df_score_202609 = df.copy()
df_score_202609.columns = df_score_202609.columns.str.strip()

resumen_p1 = pd.DataFrame({
    "periodo": [PERIODO_REF, PERIODO_ACTUAL],
    "universo_score": [len(df_score_202508), len(df_score_202609)],
    "P1": [
        df_score_202508["grupo_alerta"].astype(str).str.upper().eq("P1").sum(),
        df_score_202609["grupo_ejec"].astype(str).str.upper().eq("P1").sum(),
    ],
})
resumen_p1["pct_P1"] = resumen_p1["P1"] / resumen_p1["universo_score"] * 100
display(resumen_p1)

filas_drift = []
for variable in COLS_VARS_DIAGNOSTICO:
    fila = {"variable": variable, "estado": "No disponible", "psi": np.nan}
    if variable in df_input_202508.columns and variable in df_input_202609.columns:
        referencia = pd.to_numeric(df_input_202508[variable], errors="coerce")
        actual = pd.to_numeric(df_input_202609[variable], errors="coerce")
        fila.update({
            "estado": "OK",
            "psi": calcular_psi(referencia, actual),
            "na_202508_pct": referencia.isna().mean() * 100,
            "na_202609_pct": actual.isna().mean() * 100,
            "mediana_202508": referencia.median(),
            "mediana_202609": actual.median(),
        })
    filas_drift.append(fila)

df_drift_variables = pd.DataFrame(filas_drift).sort_values("psi", ascending=False, na_position="last")
print("Mayor drift de variables: PSI > 0.25 alto; 0.10-0.25 moderado")
display(df_drift_variables)
print(f"Input 202508: {df_input_202508.shape} | Input 202609: {df_input_202609.shape}")

,periodo,universo_score,P1,pct_P1
0,202508,161415,691,0.428089
1,202609,180309,21,0.011647


Mayor drift de variables: PSI > 0.25 alto; 0.10-0.25 moderado


,variable,estado,psi,na_202508_pct,na_202609_pct,mediana_202508,mediana_202609
16,mto_del_ext_12m,OK,5.015736,96.998792,0.000000,66219.276200,0.000000
17,rat_ing_ext_x_ing_tot_12m,OK,4.814902,96.998792,17.042221,0.293097,0.000000
20,mto_al_ext_12m,OK,4.786684,96.430762,0.000000,86596.152430,0.000000
21,ratio_egresos_exterior,OK,4.593483,96.435717,17.042221,0.227315,0.000000
23,cnt_trx_sinenv_alext_12m,OK,1.198105,93.865023,0.000000,1.000000,0.000000
28,share_cp_ingresos,OK,0.349972,95.466906,93.208956,0.000000,0.001619
26,avg_cp_men_ing_12m,OK,0.306398,95.323815,93.045904,0.000000,514.586667
31,cod_sectorista_id,OK,0.234978,89.769876,92.356540,71513.000000,25391.000000
15,share_cp_egresos,OK,0.232244,95.421687,93.184553,0.008201,0.001492
18,max_mto_cpegrmen_12m,OK,0.222673,95.323815,93.045904,3521.448000,1010.793640


Input 202508: (161435, 38) | Input 202609: (180311, 38)


In [15]:
def calcular_psi(referencia, actual, bins=10):
    referencia = pd.to_numeric(referencia, errors="coerce").dropna()
    actual = pd.to_numeric(actual, errors="coerce").dropna()

    if referencia.empty or actual.empty or referencia.nunique() < 2:
        return np.nan

    cortes = np.unique(np.nanquantile(referencia, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return np.nan

    cortes[0], cortes[-1] = -np.inf, np.inf
    ref_pct = pd.cut(
        referencia, bins=cortes, include_lowest=True
    ).value_counts(normalize=True, sort=False).clip(lower=0.0001)

    act_pct = pd.cut(
        actual, bins=cortes, include_lowest=True
    ).value_counts(normalize=True, sort=False)
    act_pct = act_pct.reindex(ref_pct.index, fill_value=0.0001).clip(lower=0.0001)

    return ((act_pct - ref_pct) * np.log(act_pct / ref_pct)).sum()